In [ ]:
# features
# 1 - MFCC 1 mean - librosa.feature.mfcc
# 2 - MFCC 2 mean
# 3 - MFCC 3 mean
# 4 - MFCC 4 mean
# 5 - MFCC 5 mean
# 6 - MFCC 6 mean
# 7 - MFCC 7 mean
# 8 - MFCC 8 mean
# 9 - MFCC 9 mean
# 10 - MFCC 10 mean
# 11 - MFCC 11 mean
# 12 - MFCC 12 mean
# 13 - MFCC 13 mean
# 14 - MFCC 1 var - librosa.feature.mfcc
# 15 - MFCC 2 var
# 16 - MFCC 3 var
# 17 - MFCC 4 var
# 18 - MFCC 5 var
# 19 - MFCC 6 var
# 20 - MFCC 7 var
# 21 - MFCC 8 var
# 22 - MFCC 9 var
# 23 - MFCC 10 var
# 24 - MFCC 11 var
# 25 - MFCC 12 var
# 26 - MFCC 13 var
# 27 - Spectral centroid mean - librosa.feature.spectral_centroid
# 28 - Spectral centroid Var
# 29 - spectral contrast - librosa.feature.spectral_contrast
# 30 - spectral rolloff - librosa.feature.spectral_rolloff
# 31 - spectral bandwiwdth - librosa.feature.spectral_bandwidth
# 32 - spectral flatness - librosa.feature.spectral_flatness
# 33 - tonnetz - librosa.feature.tonnetz
# 34 - zcr - librosa.feature.zero_crossing_rate
# 35 - rms - librosa.feature.rms
# 36 - tempo - librosa.feature.tempo
# 37 - onset?
# 38 - onset?
# 39 - onset? - Estimated tempo

# Beat strength

# Onset rate

# Onset strength

# Inter-onset interval variance

# Pulse clarity?
# 40 - beat track?
# 41 - plp?
# 42 - hpss?
# 43 - harmonic?
# 44 - percussive?
# 45 - chroma features?
# 46 - chordino?
# 47 - inharmonicity?
# 48 - dissonance?
# 49 - roughness?
# 50 - diag_strength = diag_strength = np.mean([np.mean(np.diag(ssm, k)) for k in range(1, 5)])
# intonation_var = np.var(voiced_pitches)

In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
pip install essentia

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 61.4 MB/s eta 0:00:00


In [30]:
# ===========================================
# Full Structure, Tuning, Spectral, Harmonic Feature Extraction
# ===========================================
import essentia.standard as es
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

# --- Config ---
df = pd.read_csv("/content/drive/MyDrive/data-organisation-fme-2026/all_timestamps.csv")
audio_base = "/content/drive/MyDrive/fme_audio_mp3/"

# --- TEST: first 5 rows only ---
#df = df.head(5)

sr = 44100
segment_s = 2.0  # segment length in seconds
segment_samples = int(segment_s * sr)

# --- Essentia Algorithms ---
loader = es.MonoLoader(sampleRate=sr)  # working loader
window = es.Windowing(type='hann')
spectrum = es.Spectrum()
mfcc_alg = es.MFCC(numberCoefficients=13)
pitch_alg = es.PredominantPitchMelodia(frameSize=2048, hopSize=512)
tuning_alg = es.TuningFrequency()
onset_alg = es.OnsetDetection(method='complex')

# Spectral descriptors
centroid_alg = es.SpectralCentroidTime()
rolloff_alg = es.RollOff()
flatness_alg = es.Flatness()
contrast_alg = es.SpectralContrast()
complexity_alg = es.SpectralComplexity()

results = []

for idx, row in df.iterrows():
    # Skip rows where timestamp_imputed is NaN
    if pd.isna(row['timestamp_imputed']):
        print(f"Skipping {row['drive_filename']}: timestamp_imputed is NaN")
        continue

    file_path = audio_base + row['drive_filename']
    timestamp = float(row['timestamp_imputed'])

    # --- Load audio using old working MonoLoader style ---
    try:
        audio_full = es.MonoLoader(filename=file_path, sampleRate=sr)()
    except Exception as e:
        print(f"Skipping {file_path}: {e}")
        continue

    # --- Safe segment extraction with out-of-bounds handling ---
    start = int(timestamp * sr)
    end = start + segment_samples

    if start < 0:
        start = 0
        end = min(segment_samples, len(audio_full))
    elif end > len(audio_full):
        end = len(audio_full)
        start = max(0, end - segment_samples)

    segment = audio_full[start:end]

    # Pad if shorter than segment_s
    if len(segment) < segment_samples:
        segment = np.pad(segment, (0, segment_samples - len(segment)))

    # --- Frame-wise computation ---
    spectra = []
    mfccs = []
    centroids, rolloffs, flatness_vals, contrast_vals, complexity_vals = [], [], [], [], []

    for frame in es.FrameGenerator(segment, frameSize=2048, hopSize=512):
        spec = spectrum(window(frame))
        spectra.append(spec)

        # MFCC
        bands, coeffs = mfcc_alg(spec)
        mfccs.append(coeffs)

        # Spectral descriptors
        centroids.append(centroid_alg(spec))
        rolloffs.append(rolloff_alg(spec))
        flatness_vals.append(flatness_alg(spec))
        contrast_vals.append(np.mean(contrast_alg(spec)))
        complexity_vals.append(complexity_alg(spec))

    mfccs = np.array(mfccs)

    # --- Structure / self-similarity ---
    ssm = cosine_similarity(mfccs)
    self_sim_mean = np.mean(ssm)
    self_sim_var = np.var(ssm)

    # MFCC motion & acceleration
    mfcc_delta = np.diff(mfccs, axis=0)
    mfcc_delta2 = np.diff(mfcc_delta, axis=0)
    mfcc_motion = np.mean(np.std(mfcc_delta, axis=0))
    mfcc_accel = np.mean(np.std(mfcc_delta2, axis=0))
    compression_ratio = len(np.unique(np.round(mfccs, 1), axis=0)) / len(mfccs)
    repetition_contrast = self_sim_mean / (mfcc_motion + 1e-9)
    timbre_stability = 1.0 / (mfcc_motion + 1e-9)

    # --- Spectral aggregates ---
    spectral_centroid_mean = np.mean(centroids)
    spectral_centroid_var  = np.var(centroids)
    spectral_rolloff_mean  = np.mean(rolloffs)
    spectral_flatness_mean = np.mean(flatness_vals)
    spectral_contrast_mean = np.mean(contrast_vals)
    spectral_contrast_var  = np.var(contrast_vals)
    spectral_complexity_mean = np.mean(complexity_vals)

    # --- Pitch / Tuning ---
    pitch_vals, confidence = pitch_alg(segment)
    pitch_vals = np.array(pitch_vals)
    voiced = pitch_vals[pitch_vals > 0]
    pitch_mean = np.mean(voiced) if len(voiced) > 0 else np.nan
    pitch_std  = np.std(voiced) if len(voiced) > 0 else np.nan
    pitch_range= np.max(voiced)-np.min(voiced) if len(voiced) > 0 else np.nan

    try:
        freqs, mags = es.SpectralPeaks()(segment)
        tuning_freq = tuning_alg(freqs, mags)
        tuning_dev  = 1200 * np.log2(tuning_freq / 440.0)
    except:
        tuning_freq = tuning_dev = np.nan

    # --- Onset density ---
    onset_vals = []
    prev_spec = None
    for spec in spectra:
        if prev_spec is not None:
            onset_vals.append(onset_alg(spec, prev_spec))
        prev_spec = spec
    onset_density = np.sum(np.array(onset_vals) > 0.3) / segment_s

    # --- Append Features ---
    results.append({
        "participant": row['participant'],
        "file_path": row['drive_filename'],
        "timestamp": timestamp,

        # Structure
        "self_similarity_mean": self_sim_mean,
        "self_similarity_var": self_sim_var,
        "mfcc_motion": mfcc_motion,
        "mfcc_accel": mfcc_accel,
        #"compression_ratio": compression_ratio,        #didnt work
        "repetition_contrast": repetition_contrast,
        "timbre_stability": timbre_stability,

        # Spectral
        "spectral_centroid_mean": spectral_centroid_mean,
        "spectral_centroid_var": spectral_centroid_var,
        "spectral_rolloff_mean": spectral_rolloff_mean,
        "spectral_flatness_mean": spectral_flatness_mean,
        "spectral_contrast_mean": spectral_contrast_mean,
        "spectral_contrast_var": spectral_contrast_var,
        "spectral_complexity_mean": spectral_complexity_mean,

        # Pitch / Tuning
        "pitch_mean": pitch_mean,
        "pitch_std": pitch_std,
        "pitch_range": pitch_range,
        #"tuning_frequency": tuning_freq,       #didnt work
        #"tuning_deviation_cents": tuning_dev,      #didnt work

        # Rhythm / Onsets
        "onset_density": onset_density
    })

# --- Save CSV ---
features_df = pd.DataFrame(results)
features_df.to_csv(
    "/content/drive/MyDrive/data-organisation-fme-2026/structure_tuning_features_full.csv",
    index=False
)
print(f"Saved {len(features_df)} rows to CSV")

Skipping /content/drive/MyDrive/fme_audio_mp3/song6esfnotfound.mp3: Error while configuring MonoLoader: AudioLoader: Could not open file "/content/drive/MyDrive/fme_audio_mp3/song6esfnotfound.mp3", error = No such file or directory
Skipping /content/drive/MyDrive/fme_audio_mp3/song3esfnotfound.mp3: Error while configuring MonoLoader: AudioLoader: Could not open file "/content/drive/MyDrive/fme_audio_mp3/song3esfnotfound.mp3", error = No such file or directory
Skipping /content/drive/MyDrive/fme_audio_mp3/song3esfnotfound.mp3: Error while configuring MonoLoader: AudioLoader: Could not open file "/content/drive/MyDrive/fme_audio_mp3/song3esfnotfound.mp3", error = No such file or directory
Skipping /content/drive/MyDrive/fme_audio_mp3/song3esfnotfound.mp3: Error while configuring MonoLoader: AudioLoader: Could not open file "/content/drive/MyDrive/fme_audio_mp3/song3esfnotfound.mp3", error = No such file or directory
Skipping /content/drive/MyDrive/fme_audio_mp3/song3esfnotfound.mp3: Erro

In [33]:
# ===========================================
# Full Structure, Tuning, Spectral, Harmonic Feature Extraction
# ===========================================
import essentia.standard as es
import numpy as np
import pandas as pd
import librosa
from sklearn.metrics.pairwise import cosine_similarity

# --- Config ---
df = pd.read_csv("/content/drive/MyDrive/data-organisation-fme-2026/all_timestamps.csv")
audio_base = "/content/drive/MyDrive/fme_audio_mp3/"

# --- TEST: first 5 rows only ---
#df = df.head(5)

sr = 44100
segment_s = 2.0  # segment length in seconds
segment_samples = int(segment_s * sr)

# --- Essentia Algorithms ---
loader = es.MonoLoader(sampleRate=sr)  # working loader
window = es.Windowing(type='hann')
spectrum = es.Spectrum()
mfcc_alg = es.MFCC(numberCoefficients=13)
pitch_alg = es.PredominantPitchMelodia(frameSize=2048, hopSize=512)
tuning_alg = es.TuningFrequency()
onset_alg = es.OnsetDetection(method='complex')

# Spectral descriptors
centroid_alg = es.SpectralCentroidTime()
rolloff_alg = es.RollOff()
flatness_alg = es.Flatness()
contrast_alg = es.SpectralContrast()
complexity_alg = es.SpectralComplexity()

results = []

for idx, row in df.iterrows():
    # Skip rows where timestamp_imputed is NaN
    if pd.isna(row['timestamp_imputed']):
        print(f"Skipping {row['drive_filename']}: timestamp_imputed is NaN")
        continue

    file_path = audio_base + row['drive_filename']
    timestamp = float(row['timestamp_imputed'])

    # --- Load audio using old working MonoLoader style ---
    try:
        audio_full = es.MonoLoader(filename=file_path, sampleRate=sr)()
    except Exception as e:
        print(f"Skipping {file_path}: {e}")
        continue

    # --- Safe segment extraction with out-of-bounds handling ---
    start = int(timestamp * sr)
    end = start + segment_samples

    if start < 0:
        start = 0
        end = min(segment_samples, len(audio_full))
    elif end > len(audio_full):
        end = len(audio_full)
        start = max(0, end - segment_samples)

    segment = audio_full[start:end]

    # Pad if shorter than segment_s
    if len(segment) < segment_samples:
        segment = np.pad(segment, (0, segment_samples - len(segment)))

    # --- Frame-wise computation ---
    spectra = []
    mfccs = []
    centroids, rolloffs, flatness_vals, contrast_vals, complexity_vals = [], [], [], [], []

    for frame in es.FrameGenerator(segment, frameSize=2048, hopSize=512):
        spec = spectrum(window(frame))
        spectra.append(spec)

        # MFCC
        bands, coeffs = mfcc_alg(spec)
        mfccs.append(coeffs)

        # Spectral descriptors
        centroids.append(centroid_alg(spec))
        rolloffs.append(rolloff_alg(spec))
        flatness_vals.append(flatness_alg(spec))
        contrast_vals.append(np.mean(contrast_alg(spec)))
        complexity_vals.append(complexity_alg(spec))

    mfccs = np.array(mfccs)

    # # --- Structure / self-similarity ---
    # ssm = cosine_similarity(mfccs)
    # self_sim_mean = np.mean(ssm)
    # self_sim_var = np.var(ssm)

    # # MFCC motion & acceleration
    # mfcc_delta = np.diff(mfccs, axis=0)
    # mfcc_delta2 = np.diff(mfcc_delta, axis=0)
    # mfcc_motion = np.mean(np.std(mfcc_delta, axis=0))
    # mfcc_accel = np.mean(np.std(mfcc_delta2, axis=0))
    # compression_ratio = len(np.unique(np.round(mfccs, 1), axis=0)) / len(mfccs)
    # repetition_contrast = self_sim_mean / (mfcc_motion + 1e-9)
    # timbre_stability = 1.0 / (mfcc_motion + 1e-9)

    # # --- Spectral aggregates ---
    # spectral_centroid_mean = np.mean(centroids)
    # spectral_centroid_var  = np.var(centroids)
    # spectral_rolloff_mean  = np.mean(rolloffs)
    # spectral_flatness_mean = np.mean(flatness_vals)
    # spectral_contrast_mean = np.mean(contrast_vals)
    # spectral_contrast_var  = np.var(contrast_vals)
    # spectral_complexity_mean = np.mean(complexity_vals)

    # # --- Pitch / Tuning ---
    # pitch_vals, confidence = pitch_alg(segment)
    # pitch_vals = np.array(pitch_vals)
    # voiced = pitch_vals[pitch_vals > 0]
    # pitch_mean = np.mean(voiced) if len(voiced) > 0 else np.nan
    # pitch_std  = np.std(voiced) if len(voiced) > 0 else np.nan
    # pitch_range= np.max(voiced)-np.min(voiced) if len(voiced) > 0 else np.nan

    # try:
    #     freqs, mags = es.SpectralPeaks()(segment)
    #     tuning_freq = tuning_alg(freqs, mags)
    #     tuning_dev  = 1200 * np.log2(tuning_freq / 440.0)
    # except:
    #     tuning_freq = tuning_dev = np.nan
    # --- Librosa features ---
    segment_lib = segment.astype(np.float32)
    rms = np.mean(librosa.feature.rms(y=segment_lib, frame_length=2048, hop_length=512))
    zcr = np.mean(librosa.feature.zero_crossing_rate(y=segment_lib, frame_length=2048, hop_length=512))

    # Harmonic/Percussive
    y_harm, y_perc = librosa.effects.hpss(segment_lib)
    harm_ratio = np.sum(np.abs(y_harm)) / (np.sum(np.abs(segment_lib)) + 1e-9)
    perc_ratio = np.sum(np.abs(y_perc)) / (np.sum(np.abs(segment_lib)) + 1e-9)

    # Tonnetz
    y_harm_mono = y_harm
    chroma_cens = librosa.feature.chroma_cens(y=y_harm_mono, sr=sr)
    tonnetz = librosa.feature.tonnetz(chroma=chroma_cens)
    tonnetz_mean = np.mean(tonnetz, axis=1)
    tonnetz_var  = np.var(tonnetz, axis=1)

    # MFCC 1-13 mean and var
    mfcc_mean = np.mean(mfccs, axis=0)
    mfcc_var  = np.var(mfccs, axis=0)


    # --- Append Features ---
    results.append({
        "participant": row['participant'],
        "file_path": row['drive_filename'],
        "timestamp": timestamp,

        # # Structure
        # "self_similarity_mean": self_sim_mean,
        # "self_similarity_var": self_sim_var,
        # "mfcc_motion": mfcc_motion,
        # "mfcc_accel": mfcc_accel,
        # #"compression_ratio": compression_ratio,        #didnt work
        # "repetition_contrast": repetition_contrast,
        # "timbre_stability": timbre_stability,

        # # Spectral
        # "spectral_centroid_mean": spectral_centroid_mean,
        # "spectral_centroid_var": spectral_centroid_var,
        # "spectral_rolloff_mean": spectral_rolloff_mean,
        # "spectral_flatness_mean": spectral_flatness_mean,
        # "spectral_contrast_mean": spectral_contrast_mean,
        # "spectral_contrast_var": spectral_contrast_var,
        # "spectral_complexity_mean": spectral_complexity_mean,

        # # Pitch / Tuning
        # "pitch_mean": pitch_mean,
        # "pitch_std": pitch_std,
        # "pitch_range": pitch_range,
        # #"tuning_frequency": tuning_freq,       #didnt work
        # #"tuning_deviation_cents": tuning_dev,      #didnt work

        # Librosa features
        "rms": rms,
        "zcr": zcr,
        "harmonic_ratio": harm_ratio,
        "percussive_ratio": perc_ratio,
        **{f"tonnetz_mean_{i+1}": tonnetz_mean[i] for i in range(6)},
        **{f"tonnetz_var_{i+1}": tonnetz_var[i] for i in range(6)},
        **{f"mfcc_mean_{i+1}": mfcc_mean[i] for i in range(13)},
        **{f"mfcc_var_{i+1}": mfcc_var[i] for i in range(13)},


        # # Rhythm / Onsets
        # "onset_density": onset_density
    })

# --- Save CSV ---
features_df = pd.DataFrame(results)
features_df.to_csv(
    "/content/drive/MyDrive/data-organisation-fme-2026/librosa_features_full.csv",
    index=False
)
print(f"Saved {len(features_df)} rows to CSV")

/usr/local/lib/python3.12/dist-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=1024 is too large for input signal of length=690
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/librosa/core/pitch.py:103: UserWarning: Trying to estimate tuning from empty frequency set.
  return pitch_tuning(


Skipping /content/drive/MyDrive/fme_audio_mp3/song6esfnotfound.mp3: Error while configuring MonoLoader: AudioLoader: Could not open file "/content/drive/MyDrive/fme_audio_mp3/song6esfnotfound.mp3", error = No such file or directory
Skipping /content/drive/MyDrive/fme_audio_mp3/song3esfnotfound.mp3: Error while configuring MonoLoader: AudioLoader: Could not open file "/content/drive/MyDrive/fme_audio_mp3/song3esfnotfound.mp3", error = No such file or directory
Skipping /content/drive/MyDrive/fme_audio_mp3/song3esfnotfound.mp3: Error while configuring MonoLoader: AudioLoader: Could not open file "/content/drive/MyDrive/fme_audio_mp3/song3esfnotfound.mp3", error = No such file or directory
Skipping /content/drive/MyDrive/fme_audio_mp3/song3esfnotfound.mp3: Error while configuring MonoLoader: AudioLoader: Could not open file "/content/drive/MyDrive/fme_audio_mp3/song3esfnotfound.mp3", error = No such file or directory
Skipping /content/drive/MyDrive/fme_audio_mp3/song3esfnotfound.mp3: Erro

In [40]:
# ===========================================
# Full Structure, Tuning, Spectral, Harmonic Feature Extraction
# ===========================================
import essentia.standard as es
import numpy as np
import pandas as pd
import librosa
from sklearn.metrics.pairwise import cosine_similarity

# --- Config ---
df = pd.read_csv("/content/drive/MyDrive/data-organisation-fme-2026/all_timestamps.csv")
audio_base = "/content/drive/MyDrive/fme_audio_mp3/"

# --- TEST: first 5 rows only ---
df = df.head(5)

sr = 44100
segment_s = 2.0  # segment length in seconds
segment_samples = int(segment_s * sr)

# --- Essentia Algorithms ---
loader = es.MonoLoader(sampleRate=sr)  # working loader
window = es.Windowing(type='hann')
spectrum = es.Spectrum()
mfcc_alg = es.MFCC(numberCoefficients=13)
pitch_alg = es.PredominantPitchMelodia(frameSize=2048, hopSize=512)
tuning_alg = es.TuningFrequency()
onset_alg = es.OnsetDetection(method='complex')
loader = es.MonoLoader(sampleRate=sr)
pitch_alg = es.PredominantPitchMelodia()
inharm_alg = es.Inharmonicity()
vibrato_alg = es.Vibrato()
dissonance_alg = es.Dissonance()
key_alg = es.Key()
chord_alg = es.ChordsDetection()
onset_alg = es.OnsetRate()
dance_alg = es.Danceability()

# Spectral descriptors
centroid_alg = es.SpectralCentroidTime()
rolloff_alg = es.RollOff()
flatness_alg = es.Flatness()
contrast_alg = es.SpectralContrast()
complexity_alg = es.SpectralComplexity()

spec_peaks = es.SpectralPeaks()
harm_peaks_alg = es.HarmonicPeaks()
inharm_alg = es.Inharmonicity()
pitch_alg = es.PredominantPitchMelodia(frameSize=2048, hopSize=512)
inharm_vals = []


results = []

for idx, row in df.iterrows():
    # Skip rows where timestamp_imputed is NaN
    if pd.isna(row['timestamp_imputed']):
        print(f"Skipping {row['drive_filename']}: timestamp_imputed is NaN")
        continue

    file_path = audio_base + row['drive_filename']
    timestamp = float(row['timestamp_imputed'])

    # --- Load audio using old working MonoLoader style ---
    try:
        audio_full = es.MonoLoader(filename=file_path, sampleRate=sr)()
    except Exception as e:
        print(f"Skipping {file_path}: {e}")
        continue

    # --- Safe segment extraction with out-of-bounds handling ---
    start = int(timestamp * sr)
    end = start + segment_samples

    if start < 0:
        start = 0
        end = min(segment_samples, len(audio_full))
    elif end > len(audio_full):
        end = len(audio_full)
        start = max(0, end - segment_samples)

    segment = audio_full[start:end]

    # Pad if shorter than segment_s
    if len(segment) < segment_samples:
        segment = np.pad(segment, (0, segment_samples - len(segment)))

    # --- Frame-wise features ---
    #inharm_vals, vibrato_vals, diss_vals = [], [], []

    for frame in es.FrameGenerator(segment, frameSize=2048, hopSize=512):
        # Step 1: compute fundamental pitch of frame
        # Fundamental pitch
        pitches, confidences = pitch_alg(frame)
        f0 = pitches[0] if len(pitches) > 0 else 0.0

        if f0 == 0.0:
          continue  # skip unpitched frames

        # Step 2: compute spectral peaks
        freqs, mags = spec_peaks(frame)

        # Skip if no spectral peaks or unknown pitch
        if len(freqs) == 0 or f0 == 0:
          continue

        # Step 3: compute harmonic peaks
        harm_freqs, harm_mags = harm_peaks_alg(freqs, mags, f0)

        # Step 4: compute inharmonicity
        inharm_val = inharm_alg(harm_freqs, harm_mags)
        inharm_vals.append(inharm_val)

        #vib_depth, vib_rate = vibrato_alg(frame)
        #vibrato_vals.append(vib_depth)
        #diss_vals.append(dissonance_alg(frame))

    # Aggregates
    #inharm_mean, inharm_var = np.mean(inharm_vals), np.var(inharm_vals)
    #vibrato_mean, vibrato_var = np.mean(vibrato_vals), np.var(vibrato_vals)
    #diss_mean, diss_var = np.mean(diss_vals), np.var(diss_vals)

    # Pitch / melodic range
    pitches, confidence = pitch_alg(segment)
    pitches = np.array(pitches)
    voiced = pitches[pitches > 0]
    pitch_range = np.max(voiced)-np.min(voiced) if len(voiced)>0 else np.nan
    num_notes = len(np.unique(np.round(voiced, 1))) if len(voiced)>0 else 0

    # Chords & key
    # main_chord = chords[0] if chords else None
    # num_chords = len(chords)
    segment_list = segment.tolist()  # convert numpy array to list
    try:
      chords, chord_times = chord_alg(segment_list)
    except:
      chords, chord_times = [], []

    try:
      key_name, scale = key_alg(segment_list)
    except:
      key_name, scale = None, None

    main_chord = chords[0] if chords else None
    num_chords = len(chords)

    # Harmonic / Percussive ratio
    y_harm, y_perc = librosa.effects.hpss(segment.astype(np.float32))
    hpr_ratio = np.sum(np.abs(y_harm)) / (np.sum(np.abs(segment)) + 1e-9)

    results.append({
        "participant": row['participant'],
        "file_path": row['drive_filename'],
        "timestamp": timestamp,
        #"inharm_mean": inharm_mean,
        #"inharm_var": inharm_var,
        #"vibrato_mean": vibrato_mean,
        #"vibrato_var": vibrato_var,
        #"dissonance_mean": diss_mean,
        #"dissonance_var": diss_var,
        "pitch_range": pitch_range,
        "num_notes": num_notes,
        #"main_chord": main_chord,
        #"num_chords": num_chords,
        #"key": key_name,
        #"scale": scale,
        "harmonic_percussive_ratio": hpr_ratio
    })

# --- Save CSV ---
features_df = pd.DataFrame(results)
features_df.to_csv(
    "/content/drive/MyDrive/data-organisation-fme-2026/else_features_full.csv",
    index=False
)
print(f"Saved {len(features_df)} rows to CSV")

/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:3596: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:138: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:4008: RuntimeWarning: Degrees of freedom <= 0 for slice
  return _methods._var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:175: RuntimeWarning: invalid value encountered in divide
  arrmean = um.true_divide(arrmean, div, out=arrmean,
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:210: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


Saved 5 rows to CSV


In [14]:
# ===========================================
# CLEAN FEATURE EXTRACTION (TEST VERSION)
# ===========================================

import essentia.standard as es
import numpy as np
import pandas as pd
import librosa
import os

# --- Config ---
df = pd.read_csv("/content/drive/MyDrive/data-organisation-fme-2026/all_timestamps.csv")
audio_base = "/content/drive/MyDrive/fme_audio_mp3/"
output_csv = "/content/drive/MyDrive/data-organisation-fme-2026/more_features_700-1000.csv"

# TEST ONLY
#df = df.head(100)
df = df.iloc[1400:]


sr = 44100
segment_s = 2.0
frame_size = 2048
hop_size = 1024

# ===========================================
# KEY ENCODING
# ===========================================

KEY_MAP = {
"C":0,"C#":1,"Db":1,
"D":2,"D#":3,"Eb":3,
"E":4,
"F":5,"F#":6,"Gb":6,
"G":7,"G#":8,"Ab":8,
"A":9,"A#":10,"Bb":10,
"B":11
}

SCALE_MAP = {
"major":1,
"minor":0
}

# ===========================================
# ALGORITHMS
# ===========================================

window = es.Windowing(type='hann')
spectrum = es.Spectrum()

spectral_peaks = es.SpectralPeaks(
    sampleRate=sr,
    magnitudeThreshold=1e-6,
    minFrequency=50,
    maxFrequency=sr/2
)

inharm_alg = es.Inharmonicity()
dissonance_alg = es.Dissonance()

rhythm_extractor = es.RhythmExtractor2013(method='multifeature')
onset_rate_alg = es.OnsetRate()

key_extractor = es.KeyExtractor()

pitch_yinfft = es.PitchYinFFT(frameSize=frame_size, sampleRate=sr)
pitch_salience_alg = es.PitchSalience(sampleRate=sr)

melody_alg = es.PredominantPitchMelodia(
    frameSize=frame_size,
    hopSize=hop_size,
    minFrequency=50,
    maxFrequency=5000
)

loader = es.MonoLoader(sampleRate=sr)

results = []

# ===========================================
# LOOP
# ===========================================

for idx,row in df.iterrows():

    file_path = os.path.join(audio_base,row['drive_filename'])
    timestamp = float(row['timestamp_imputed'])

    try:
        audio_full = es.MonoLoader(filename=file_path,sampleRate=sr)()
    except:
        print("Skipping",file_path)
        continue

    start = int(max(0,(timestamp-1)*sr))
    end = int(min(len(audio_full),(timestamp+1)*sr))
    segment = audio_full[start:end]

    if len(segment) < segment_s*sr:
        segment = np.pad(segment,(0,int(segment_s*sr)-len(segment)))

    # ===================================
    # Rhythm
    # ===================================

    bpm,beats,beats_conf,_,danceability = rhythm_extractor(segment)

    num_beats = len(beats)

    # FIX DANCEABILITY
    danceability = float(np.mean(danceability))

    # FIX ONSET RATE
    onset_times,onset_rate = onset_rate_alg(segment)
    onset_rate = float(onset_rate)

    # ===================================
    # Key
    # ===================================

    key,scale,key_strength = key_extractor(segment)

    key_numeric = KEY_MAP.get(key,np.nan)
    scale_numeric = SCALE_MAP.get(scale,np.nan)

    # ===================================
    # Melody
    # ===================================

    pitches,_ = melody_alg(segment)
    pitches = np.array(pitches)

    voiced = pitches[pitches>0]

    pitch_range = np.max(voiced)-np.min(voiced) if len(voiced)>0 else np.nan
    num_notes = len(np.unique(np.round(voiced,1))) if len(voiced)>0 else 0
    vibrato_extent = np.std(voiced) if len(voiced)>0 else np.nan

    # ===================================
    # Frame analysis
    # ===================================


    # ===================================
    # Harmonic / Percussive
    # ===================================

    y_harm,y_perc = librosa.effects.hpss(segment.astype(np.float32))
    hpr_ratio = np.sum(np.abs(y_harm))/(np.sum(np.abs(segment))+1e-9)

    # ===================================
    # STORE
    # ===================================

    results.append({

        "file_path":row['drive_filename'],

        "bpm":bpm,
        "num_beats":num_beats,
        "onset_rate":onset_rate,
        "danceability":danceability,

        "key":key_numeric,
        "scale":scale_numeric,
        "key_strength":key_strength,

        #"pitch_range":pitch_range,
        #"num_notes":num_notes,
        #"pitch_yin":pitch_yin,
        #"pitch_salience":pitch_salience,
        #"vibrato_extent":vibrato_extent,

        #"inharm_mean":inharm_mean,
        #"inharm_var":inharm_var,

        #"dissonance_mean":diss_mean,
        #"dissonance_var":diss_var,

        #"harmonic_percussive_ratio":hpr_ratio
    })

# ===========================================
# SAVE
# ===========================================

features_df = pd.DataFrame(results)
features_df.to_csv(output_csv,index=False)

print("Saved",len(features_df),"rows")
print(features_df.head())

Skipping /content/drive/MyDrive/fme_audio_mp3/song2esfnotfound.mp3
Skipping /content/drive/MyDrive/fme_audio_mp3/song2esfnotfound.mp3
Saved 390 rows
                                           file_path         bpm  num_beats  \
0       solomonhanszimmer12yearsaslavesample2mp3.mp3   98.446434          3   
1  tenetrainynightintallinnludwiggoranssonsample1...  120.185326          3   
2  tenetrainynightintallinnludwiggoranssonsample1...  114.843758          4   
3  tenetrainynightintallinnludwiggoranssonsample1...  117.453842          3   
4  trappedairaaroncupplesislandofthehungryghostss...  143.554688          4   

   onset_rate  danceability  key  scale  key_strength  
0         1.0      0.609524   11      1      0.834414  
1         0.0      0.499229   10      1      0.695043  
2         0.0      0.491489    3      0      0.483954  
3         0.0      0.487619    3      0      0.376933  
4         0.0      0.445049    4      0      0.740815  


In [15]:
import pandas as pd
import numpy as np

# =====================================
# LOAD DATASET
# =====================================

path = "/content/drive/MyDrive/data-organisation-fme-2026/librosa_features_full_fme_dataset_MARCH_2026.csv"

df = pd.read_csv(path)

# =====================================
# CHORD ENCODING MAPS
# =====================================

ROOT_MAP = {
"C":0,"C#":1,"Db":1,
"D":2,"D#":3,"Eb":3,
"E":4,
"F":5,"F#":6,"Gb":6,
"G":7,"G#":8,"Ab":8,
"A":9,"A#":10,"Bb":10,
"B":11
}

TYPE_MAP = {
"major":0,
"minor":1,
"diminished":2,
"augmented":3
}

# =====================================
# CHORD PARSER
# =====================================

def parse_chord(chord):

    if pd.isna(chord):
        return np.nan, np.nan

    chord = chord.strip()

    # detect root (handles sharps/flats)
    if len(chord) >= 2 and chord[1] in ["#","b"]:
        root = chord[:2]
        quality = chord[2:]
    else:
        root = chord[:1]
        quality = chord[1:]

    root_num = ROOT_MAP.get(root,np.nan)

    # detect chord quality
    if quality in ["","maj"]:
        type_num = TYPE_MAP["major"]
    elif quality in ["m","min"]:
        type_num = TYPE_MAP["minor"]
    elif "dim" in quality:
        type_num = TYPE_MAP["diminished"]
    elif "aug" in quality:
        type_num = TYPE_MAP["augmented"]
    else:
        type_num = np.nan

    return root_num, type_num

# =====================================
# APPLY CONVERSION
# =====================================

roots = []
types = []

for chord in df["main_chord"]:
    r,t = parse_chord(chord)
    roots.append(r)
    types.append(t)

df["chord_root"] = roots
df["chord_type"] = types

# optional: drop original string column
df = df.drop(columns=["main_chord"])

# =====================================
# SAVE NEW CSV
# =====================================

output = "/content/drive/MyDrive/data-organisation-fme-2026/librosa_features_full_fme_dataset_MARCH_2026_NUMERIC_CHORDS.csv"

df.to_csv(output,index=False)

print("Saved new dataset:",output)

Saved new dataset: /content/drive/MyDrive/data-organisation-fme-2026/librosa_features_full_fme_dataset_MARCH_2026_NUMERIC_CHORDS.csv
